# Reddit Kaggle Competition: Inference Notebook

Loads the trained BiLSTM + Linear Attention model and generates predictions on the test set.

**Metric:** Macro ROC AUC.

In [ ]:
import json
import os
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

DATA_DIR = '../data/'
MODEL_PATH = 'model_best.pth'
VOCAB_PATH = 'vocab.json'
BATCH_SIZE = 128
MAX_LEN = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. Load the Data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_kaggle.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_kaggle.csv'))

emotion_cols = [c for c in train_df.columns if c not in ['ID', 'text']]
num_labels = len(emotion_cols)

print(f"Test samples: {len(test_df)}")
print(f"Total emotions: {num_labels}")

## 2. Tokenization & Vocabulary
Same tokenization scheme as training; vocab is loaded from `vocab.json` produced by the training run.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

with open(VOCAB_PATH, 'r') as f:
    vocab = json.load(f)
vocab_size = len(vocab)

def text_to_ids(text):
    ids = [vocab.get(w, vocab.get('<UNK>', 1)) for w in tokenize(text)][:MAX_LEN]
    if not ids:
        ids = [vocab.get('<UNK>', 1)]
    return ids + [0] * (MAX_LEN - len(ids))

print(f"Vocabulary size: {vocab_size}")

## 3. Dataset and DataLoader

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].apply(text_to_ids).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long)

test_loader = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

## 4. Model Definition & Loading
Same BiLSTM + Linear Attention used during training.

In [ ]:
class LinearAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, mask):
        safe_mask = mask.clone()
        safe_mask[:, 0] |= ~mask.any(dim=1)
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(~safe_mask, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * lstm_out).sum(dim=1)

class BiLSTMAttentionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_dim=256, output_dim=28,
                 dropout_p=0.4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout_p)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_p if num_layers > 1 else 0.0,
        )
        self.attention = LinearAttention(hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        mask = x != 0
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        context = self.attention(lstm_out, mask)
        return self.fc(self.dropout(context))

model = BiLSTMAttentionModel(vocab_size, output_dim=num_labels).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print('Model loaded and ready for inference.')

## 5. `predict_emotion` Function
Single-text helper used by the grader.

In [ ]:
def predict_emotion(text):
    ids = torch.tensor([text_to_ids(text)], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(ids)).cpu().numpy()[0]
    return dict(zip(emotion_cols, probs.tolist()))

## 6. Generate Submission

In [ ]:
probs = []
with torch.no_grad():
    for x in test_loader:
        probs.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())

submission = pd.DataFrame(np.vstack(probs), columns=emotion_cols)
submission.insert(0, 'ID', test_df['ID'].values)
submission.to_csv('submission.csv', index=False)
print('Submission file created: submission.csv')